# Solar Flare ML Prediction - ML Pipeline and Evaluation

This notebook demonstrates the complete machine learning pipeline with model training, evaluation, and performance metrics.

## 1. Import Libraries and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc, accuracy_score,
    classification_report, brier_score_loss
)
# Machine Learning algorithms
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from scipy import interp
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Load and Prepare Data

In [ ]:
# Load training data (Solar Cycle 22)
names = ['mcint', 'mcint_evol', 'class']
df_train = pd.read_csv('../mcint_ml22.csv', names=names,
                        dtype={'mcint': str, 'mcint_evol': str, 'class': np.float64})

# Load test data (Solar Cycle 23)
df_test = pd.read_csv('../mcint_ml23.csv', names=names,
                       dtype={'mcint': str, 'mcint_evol': str, 'class': np.float64})

# Feature engineering - parse evolution codes
le = preprocessing.LabelEncoder()
df_train['mcint_enc'] = le.fit_transform(df_train['mcint'])
df_train['mcint_evol_enc'] = le.fit_transform(df_train['mcint_evol'])
df_train['z1'] = df_train['mcint_evol'].str[0]
df_train['p1'] = df_train['mcint_evol'].str[1]
df_train['c1'] = df_train['mcint_evol'].str[2]
df_train['z2'] = df_train['mcint_evol'].str[3]
df_train['p2'] = df_train['mcint_evol'].str[4]
df_train['c2'] = df_train['mcint_evol'].str[5]

df_test['mcint_enc'] = le.fit_transform(df_test['mcint'])
df_test['mcint_evol_enc'] = le.fit_transform(df_test['mcint_evol'])
df_test['z1'] = df_test['mcint_evol'].str[0]
df_test['p1'] = df_test['mcint_evol'].str[1]
df_test['c1'] = df_test['mcint_evol'].str[2]
df_test['z2'] = df_test['mcint_evol'].str[3]
df_test['p2'] = df_test['mcint_evol'].str[4]
df_test['c2'] = df_test['mcint_evol'].str[5]

print(f"Training data shape: {df_train.shape}")
print(f"Test data shape: {df_test.shape}")

## 3. Custom Skill Scores

The project uses two custom skill scores:
- **Brier Skill Score (BSS)**: Probabilistic skill score
- **True Skill Statistic (TSS)**: Binary classification skill metric

In [ ]:
# Define custom skill score functions
def tss_calc(cont_table):
    """
    Calculate True Skill Statistic (TSS)
    TSS = POD - POFD
    where POD = True Positive Rate, POFD = False Positive Rate
    Range: [-1, 1], Higher is better
    """
    TP = cont_table[1, 1]
    TN = cont_table[0, 0]
    FP = cont_table[0, 1]
    FN = cont_table[1, 0]
    
    POD = TP / (TP + FN)  # True Positive Rate
    POFD = FP / (TN + FP)  # False Positive Rate
    TSS = POD - POFD
    return TSS, POD, POFD

def bss_calc(Y_val, predict_probs):
    """
    Calculate Brier Skill Score (BSS)
    BSS = 1 - (BS / BS_clim)
    where BS = Brier Score, BS_clim = climatological Brier Score
    Range: [-∞, 1], Higher is better
    """
    clim_arr = np.full(len(Y_val), np.mean(Y_val))
    bs_metric = brier_score_loss(Y_val, predict_probs)
    bs_metric_clim = brier_score_loss(Y_val, clim_arr)
    return 1. - bs_metric / bs_metric_clim

print("Custom Skill Score Functions Defined")
print("\nBrier Skill Score (BSS):")
print("  - Compares probabilistic predictions against climatological baseline")
print("  - Range: [-∞, 1] (Higher is better)")
print("  - 0.0 = Same as climatology")
print("  - >0.0 = Better than climatology")
print("\nTrue Skill Statistic (TSS):")
print("  - Balances sensitivity and specificity")
print("  - Range: [-1, 1] (Higher is better)")
print("  - 0.0 = No skill (random guessing)")
print("  - 1.0 = Perfect prediction")

## 4. Prepare Data for Modeling

Select features for the "sep_zpc" method (separate Zurich-Penetration-Class components)

In [ ]:
# Feature selection: Use parsed evolution code components
feature_method = 'sep_zpc'  # Use separated ZPC components
X_train = df_train[['z1', 'p1', 'c1', 'z2', 'p2', 'c2']]
Y_train = df_train['class']
X_test = df_test[['z1', 'p1', 'c1', 'z2', 'p2', 'c2']]
Y_test = df_test['class']

print(f"Feature set: {feature_method}")
print(f"Features: {X_train.columns.tolist()}")
print(f"\nTraining set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test set: {X_test.shape[0]} samples, {X_test.shape[1]} features")
print(f"\nClass distribution (Train):")
print(Y_train.value_counts().sort_index())
print(f"\nClass distribution (Test):")
print(Y_test.value_counts().sort_index())

## 5. K-Fold Cross-Validation Pipeline

In [ ]:
# Define machine learning models
models = [
    ('LR', LogisticRegression(max_iter=1000)),
    ('LDA', LinearDiscriminantAnalysis()),
    ('KNN', KNeighborsClassifier()),
    ('CART', DecisionTreeClassifier()),
    ('RFC', RandomForestClassifier(n_estimators=100, random_state=42))
]

# K-Fold Cross-Validation setup
cv = StratifiedKFold(n_splits=10, shuffle=False, random_state=42)

# Storage for results
results_bss = {}
results_tss = {}
results_auc = {}
results_accuracy = {}

# Colors for visualization
colors_models = ['pink', 'lightblue', 'lightgreen', 'peachpuff', 'mediumpurple']

print("Starting K-Fold Cross-Validation with 10 folds...\n")

# Prepare data for sklearn
x_arr = X_train.values
y_arr = Y_train.values

# Evaluate each model
for name, model in models:
    print(f"\nEvaluating: {name}")
    
    bss_scores = []
    tss_scores = []
    auc_scores = []
    accuracy_scores = []
    
    fold = 1
    for train_idx, test_idx in cv.split(x_arr, y_arr):
        # Split data
        X_fold_train, X_fold_test = x_arr[train_idx], x_arr[test_idx]
        y_fold_train, y_fold_test = y_arr[train_idx], y_arr[test_idx]
        
        # Train and predict
        model.fit(X_fold_train, y_fold_train)
        probs = model.predict_proba(X_fold_test)
        predicted = model.predict(X_fold_test)
        
        # Calculate metrics
        conf_mat = confusion_matrix(y_fold_test, predicted)
        tss, pod, pofd = tss_calc(conf_mat)
        bss = bss_calc(y_fold_test, probs[:, 1])
        fpr, tpr, _ = roc_curve(y_fold_test, probs[:, 1])
        auc_score = auc(fpr, tpr)
        acc = accuracy_score(y_fold_test, predicted)
        
        bss_scores.append(bss)
        tss_scores.append(tss)
        auc_scores.append(auc_score)
        accuracy_scores.append(acc)
        
        fold += 1
    
    # Store results
    results_bss[name] = bss_scores
    results_tss[name] = tss_scores
    results_auc[name] = auc_scores
    results_accuracy[name] = accuracy_scores
    
    # Print fold results
    print(f"  Mean BSS: {np.mean(bss_scores):.4f} ± {np.std(bss_scores):.4f}")
    print(f"  Mean TSS: {np.mean(tss_scores):.4f} ± {np.std(tss_scores):.4f}")
    print(f"  Mean AUC: {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
    print(f"  Mean Accuracy: {np.mean(accuracy_scores):.4f} ± {np.std(accuracy_scores):.4f}")

## 6. Performance Comparison - Brier Skill Score

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

bss_data = [results_bss[name] for name, _ in models]
model_names = [name for name, _ in models]

bp = ax.boxplot(bss_data, labels=model_names, patch_artist=True, widths=0.6)

# Color boxes
for patch, color in zip(bp['boxes'], colors_models):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Add mean markers
means_bss = [np.mean(scores) for scores in bss_data]
ax.scatter(range(1, len(model_names) + 1), means_bss, color='red', s=100, zorder=3, marker='D', label='Mean')

# Add baseline reference
ax.axhline(y=0.09, color='green', linestyle=':', linewidth=2, label='Uncorrected Poisson BSS', alpha=0.7)
ax.axhline(y=0.20, color='darkgreen', linestyle='--', linewidth=2, label='Corrected Poisson BSS', alpha=0.7)

ax.set_ylabel('Brier Skill Score (BSS)', fontsize=12, fontweight='bold')
ax.set_xlabel('Algorithm', fontsize=12, fontweight='bold')
ax.set_title('Algorithm Performance Comparison - Brier Skill Score (K-Fold CV)', fontsize=13, fontweight='bold')
ax.set_ylim([-0.05, 0.4])
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

best_bss = model_names[np.argmax(means_bss)]
ax.text(0.05, 0.95, f'Best: {best_bss} (BSS={max(means_bss):.4f})',
        transform=ax.transAxes, fontsize=11, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("\nBrier Skill Score Summary:")
for name, scores in results_bss.items():
    print(f"  {name}: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

## 7. Performance Comparison - True Skill Statistic

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

tss_data = [results_tss[name] for name, _ in models]

bp = ax.boxplot(tss_data, labels=model_names, patch_artist=True, widths=0.6)

# Color boxes
for patch, color in zip(bp['boxes'], colors_models):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Add mean markers
means_tss = [np.mean(scores) for scores in tss_data]
ax.scatter(range(1, len(model_names) + 1), means_tss, color='red', s=100, zorder=3, marker='D', label='Mean')

# Add baseline reference
ax.axhline(y=0.45, color='darkgreen', linestyle='--', linewidth=2, label='Poisson TSS', alpha=0.7)

ax.set_ylabel('True Skill Statistic (TSS)', fontsize=12, fontweight='bold')
ax.set_xlabel('Algorithm', fontsize=12, fontweight='bold')
ax.set_title('Algorithm Performance Comparison - True Skill Statistic (K-Fold CV)', fontsize=13, fontweight='bold')
ax.set_ylim([0.3, 0.8])
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

best_tss = model_names[np.argmax(means_tss)]
ax.text(0.05, 0.05, f'Best: {best_tss} (TSS={max(means_tss):.4f})',
        transform=ax.transAxes, fontsize=11, verticalalignment='bottom',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("\nTrue Skill Statistic Summary:")
for name, scores in results_tss.items():
    print(f"  {name}: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

## 8. AUC-ROC Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

auc_data = [results_auc[name] for name, _ in models]

bp = ax.boxplot(auc_data, labels=model_names, patch_artist=True, widths=0.6)

# Color boxes
for patch, color in zip(bp['boxes'], colors_models):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Add mean markers
means_auc = [np.mean(scores) for scores in auc_data]
ax.scatter(range(1, len(model_names) + 1), means_auc, color='red', s=100, zorder=3, marker='D', label='Mean')

ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=2, label='Random Classifier', alpha=0.7)

ax.set_ylabel('AUC-ROC Score', fontsize=12, fontweight='bold')
ax.set_xlabel('Algorithm', fontsize=12, fontweight='bold')
ax.set_title('Algorithm Performance Comparison - AUC-ROC (K-Fold CV)', fontsize=13, fontweight='bold')
ax.set_ylim([0.4, 1.0])
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

best_auc = model_names[np.argmax(means_auc)]
ax.text(0.05, 0.05, f'Best: {best_auc} (AUC={max(means_auc):.4f})',
        transform=ax.transAxes, fontsize=11, verticalalignment='bottom',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("\nAUC-ROC Summary:")
for name, scores in results_auc.items():
    print(f"  {name}: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

## 9. Multi-Metric Comparison Table

In [ ]:
# Create comprehensive performance table
summary_data = []
for name, _ in models:
    summary_data.append({
        'Algorithm': name,
        'BSS': f"{np.mean(results_bss[name]):.4f} ± {np.std(results_bss[name]):.4f}",
        'TSS': f"{np.mean(results_tss[name]):.4f} ± {np.std(results_tss[name]):.4f}",
        'AUC-ROC': f"{np.mean(results_auc[name]):.4f} ± {np.std(results_auc[name]):.4f}",
        'Accuracy': f"{np.mean(results_accuracy[name]):.4f} ± {np.std(results_accuracy[name]):.4f}"
    })

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*100)
print("K-FOLD CROSS-VALIDATION RESULTS (10 Folds)")
print("="*100)
print(summary_df.to_string(index=False))
print("="*100)
print("\nNote: Values shown as Mean ± Standard Deviation")
print("\nInterpretation:")
print("  - BSS: Higher is better (>0.20 indicates operational skill)")
print("  - TSS: Higher is better (range -1 to 1)")
print("  - AUC-ROC: Higher is better (0.5=random, 1.0=perfect)")
print("  - Accuracy: Higher is better (but may be misleading for imbalanced classes)")